# Fine-tuning de Qwen 2.5 3B para clasificación de relevancia RAG

## Objetivo del notebook

Este notebook implementa el **fine-tuning con QLoRA** del modelo Qwen 2.5 3B-Instruct para la tarea de
**clasificación de relevancia de documentos** en el pipeline RAG de NormaBot.

El modelo aprenderá a decidir, dado un par `(consulta, documento)`, si el documento contiene
información útil para responder la consulta. La salida es binaria:
- **`relevante`** → el documento debe incluirse en el contexto de generación
- **`no relevante`** → el documento debe descartarse

## Contexto en NormaBot

El pipeline RAG (`src/rag/main.py`) sigue el flujo:

```
Retrieve (ChromaDB) → Grade (Qwen 2.5 3B via Ollama) → Generate (Bedrock Nova Lite)
```

La etapa de **grading** actual usa el modelo base con prompting directo. Este fine-tuning busca
especializarlo en el dominio legal (EU AI Act, BOE) para reducir falsos positivos en el contexto
de generación. El experimento:

1. Establece un **baseline** con el modelo base + prompting
2. **Fine-tunea** el mismo modelo con QLoRA sobre el dataset de relevancia
3. **Compara** ambos enfoques: Accuracy, Precision, Recall, F1-macro
4. Proporciona el **código de integración** listo para `src/rag/main.py`

## Preparación en Colab

Antes de ejecutar, copia `grading_dataset.jsonl` al root de Colab (`/content/`) y
monta Google Drive en `/content/drive/` para persistir el modelo entrenado.


# Instalación de librerías y comprobación de entorno


In [ ]:
%%capture
# Desinstalar versiones conflictivas si las hay
!pip uninstall -y torch torchvision torchaudio bitsandbytes transformers peft trl

# PyTorch con soporte CUDA 12.1 (compatible con Colab T4 y A10G)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Stack de fine-tuning
!pip install \
    transformers==4.46.3 \
    tokenizers \
    accelerate \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    datasets

# Métricas y tracking
!pip install scikit-learn mlflow python-dotenv pandas

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# En Colab: subir functions.py a /content/ junto con grading_dataset.jsonl
sys.path.insert(0, "/content")
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

# Dataset de relevancia

El dataset está generado en `data/processed/grading_dataset.jsonl` mediante
`data/generate_grading_dataset.py`. Cada línea es un ejemplo con tres campos:

```json
{"query": "¿Qué sistemas de IA están prohibidos?", "document": "Artículo 5...", "label": "relevante"}
{"query": "¿Qué sistemas de IA están prohibidos?", "document": "Real Decreto 463/2020...", "label": "no relevante"}
```

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `query` | str | Consulta del usuario sobre normativa de IA |
| `document` | str | Fragmento de texto recuperado de ChromaDB |
| `label` | str | `"relevante"` o `"no relevante"` |

**Composición:** 117 queries · 634 ejemplos · 44.6% relevantes / 55.4% no relevantes  
**Fuentes:** EU AI Act (Arts. 5, 6, 9–17, 25–26, 43, 47–53, 72–73, 99, 113), AESIA, RGPD/LOPDGDD  
**Negativos:** hard negatives del dominio (otro artículo legal) + easy negatives (otra ley ajena al dominio)


In [ ]:
# ============================================================
# MONTAR GOOGLE DRIVE (para guardar el modelo de forma persistente)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CONFIGURACIÓN DE RUTAS Y CONSTANTES
# ============================================================

# Dataset — copiar grading_dataset.jsonl al root de Colab antes de ejecutar
DATASET_PATH = Path("/content/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Salida en Google Drive — persiste entre sesiones de Colab
DRIVE_BASE   = Path("/content/drive/MyDrive/normabot")
OUTPUT_DIR   = str(DRIVE_BASE / "qwen-grader-lora")
ADAPTER_PATH = str(DRIVE_BASE / "qwen-grader-lora" / "adapter_final")
MERGED_PATH  = str(DRIVE_BASE / "qwen-grader-merged")   # solo si se hace merge

MAX_SEQ_LEN = 512

print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

In [ ]:
all_data = load_grading_dataset(DATASET_PATH)

In [ ]:
train_data, val_data, test_data = split_dataset(all_data)

In [ ]:
show_split_stats(train_data, "TRAIN")
show_split_stats(val_data,   "VALIDATION")
show_split_stats(test_data,  "TEST")

print(f"\n{'='*55}")
print("Ejemplo del dataset:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

# Formato del prompt de entrenamiento

Para el fine-tuning usamos el **chat template nativo de Qwen 2.5** (formato `<|im_start|>`).
Cada ejemplo de entrenamiento tiene la forma:

```
<|im_start|>system
Eres un asistente especializado en normativa de IA...<|im_end|>
<|im_start|>user
Consulta: {query}

Documento: {document}

¿Es este documento relevante para responder la consulta?<|im_end|>
<|im_start|>assistant
{label}<|im_end|>
```

El modelo aprende a generar directamente `relevante` o `no relevante` como turno de assistant.


In [ ]:
# Carga minima del tokenizador para visualizar el formato del prompt
tokenizer = load_tokenizer(MODEL_ID)

sample_formatted = format_training_prompt(train_data[0], tokenizer)
print("Ejemplo de prompt de entrenamiento:")
print("-" * 65)
print(sample_formatted)
print("-" * 65)
print(f"Longitud: {len(sample_formatted)} chars")

In [ ]:
train_dataset = examples_to_hf_dataset(train_data, tokenizer)
val_dataset   = examples_to_hf_dataset(val_data,   tokenizer)

print(f"Dataset de entrenamiento: {len(train_dataset)} ejemplos")
print(f"Dataset de validacion:    {len(val_dataset)} ejemplos")
print()
print("Primeros 300 chars del primer ejemplo de train:")
print(train_dataset[0]["text"][:300])